# EMERGING parser walkthrough

This notebook is the practical guide for parsing the EMERGING MATLAB bundles supported by MARIO.


## What this notebook covers

- which official Zenodo version records are relevant for MARIO;
- how local file naming works for `v1.0` and `v2.x` bundles;
- how `download_emerging(...)` maps to the supported version records;
- how `year=`, `regions=`, `load_co2=`, and `co2_path=` are used;
- which caveats matter for very large EMERGING databases.


## Supported official version records

The relevant official records are:

- `v2.2`: [Zenodo 19461860](https://doi.org/10.5281/zenodo.19461860)
- `v2.0`: [Zenodo 17557778](https://doi.org/10.5281/zenodo.17557778)
- `v1.0`: [Zenodo 10956623](https://doi.org/10.5281/zenodo.10956623)

Any other version reference should be treated as deprecated in the MARIO documentation.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_emerging(...)`

The parser currently supports only the multiregional `IOT` workflow.


## Key arguments

The key public arguments are:

- `path`: one EMERGING `.mat` file or a directory containing multiple yearly bundles;
- `table`: currently only `"IOT"` is supported;
- `year`: use it when one directory contains more than one EMERGING year;
- `regions`: optional ISO3 subset to keep only one manageable part of the database;
- `load_co2`: enable or disable automatic companion CO2 import;
- `co2_path`: explicit path to one companion CO2 file when auto-detection is not enough.


## Local file naming conventions

In practice MARIO accepts these local naming conventions:

- `global_mrio_<year>.mat` for `v1.0`;
- `EMERGING_V2_<year>_m.mat` for `v2.x`;
- `EMERGING_V2_<year>.mat` for older local `v2.x` copies when the internal MATLAB structure is still compatible.

For local `v2.x` files, MARIO does not try to infer the exact sub-version `2.0` versus `2.1` versus `2.2` from the filename alone, because the public naming convention is shared across those releases.


## Expected path structure

`path` can point either to one yearly `.mat` file or to a directory containing the yearly EMERGING bundle and, optionally, the companion CO2 file.

Typical directory layout:

```text
EMERGING/2.2/
├── EMERGING_V2_2023_m.mat
└── EMERGING_CO2_2023.mat
```

For the older `v1.0` naming convention, the economic bundle is typically named like `global_mrio_<year>.mat`.

When `path` is a directory, use `year=` to select the economic bundle. With `load_co2=True`, MARIO first looks for the matching CO2 file in the same directory, for example `EMERGING_CO2_2023.mat` for `year=2023`. Use `co2_path=` when the companion file is stored elsewhere or has a non-standard name.

## Download workflow

Use `mario.download_emerging(...)` when you want MARIO to fetch one of the supported official version records.

`latest` currently resolves to `v2.2`.


In [1]:
import mario

## Download one official version explicitly


In [2]:
mario.download_emerging(
    path="/path/to/2.2",
    version="2.2",
    years=[2023],
)

{'source': 'https://doi.org/10.5281/zenodo.10956622',
 'version': '2.2',
 'version_record': 'https://doi.org/10.5281/zenodo.19461860',
 'download_dir': '/path/to/emerging_directory',
 'years': [2023],
 'files': ['/path/to/emerging_directory/EMERGING_CO2_2023.mat',
  '/path/to/EMERGING_V2_2023_m.mat']}

## Parse one local EMERGING bundle


In [2]:
db = mario.parse_emerging(
    path="/path/to/EMERGING_V2_2023_m.mat",
    table="IOT",
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Parse from a directory and select one year


In [3]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Restrict the region set

The full EMERGING matrix is very large, so `regions=` is often the right first step.


In [4]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
    regions=["DEU", "FRA", "ITA"],
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 3 regions, 133 sectors, 9 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 3
Sector = 133

## Control CO2 loading explicitly


In [5]:
db = mario.parse_emerging(
    path="/path/to/2.2",
    table="IOT",
    year=2023,
    co2_path="/path/to/EMERGING_CO2_2023.mat",
)

db

INFO Parser: reading EMERGING bundle EMERGING_V2_2023_m.mat.


INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2023.mat.


INFO Parser: EMERGING parsed with 245 regions, 133 sectors, 735 final-demand columns and 7 satellite rows.


INFO Metadata: initialized.


name = EMERGING 2023
table = IOT
scenarios = ['baseline']
Factor of production = 1
Satellite account = 7
Consumption category = 3
Region = 245
Sector = 133

## Caveats

- EMERGING parsing currently supports only `IOT` tables;
- local `v2.x` file names do not identify the exact official sub-version;
- `regions=` is often necessary to keep the database manageable;
- `load_co2=False` is useful when you want to parse the core IOT first and deal with extensions separately.


In [6]:
db.table_type

'IOT'

In [7]:
db.get_index("Region")[:10]

['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ANT', 'BES', 'CUW', 'ARE']

In [8]:
db.get_index("Satellite account")

['Coal',
 'Natural gas',
 'Oil products',
 'Crude, NGL, Ref Feeds.',
 'Other',
 'Oil shale & oil sands',
 'Peat & Peat products']

In [9]:
db.meta.source

'EMERGING MATLAB bundle via Zenodo; (concept DOI https://doi.org/10.5281/zenodo.10956622); Huo, J., Chen, P., Hubacek, K., Zheng, H., Meng, J., & Guan, D. (2022). Full-scale, near real-time multi-regional input-output table for the global emerging economies (EMERGING). Journal of Industrial Ecology, 26, 1218–1232. https://doi.org/10.1111/jiec.13264'